# ResSENet: Residual Squeeze-and-Excitation Networks for Robust Facial Emotion Recognition

**Publication-Grade Implementation | FER2013 Benchmark**

This notebook reproduces all results from the paper:
- Table I: Comparative Evaluation (6 baselines + ResSENet + ResSENet+)
- Table II: Ablation Study (4 variants)
- Figure: Per-class confusion matrix
- Figure: GradCAM interpretability visualization
- Figure: Accuracy-Parameter trade-off plot

**Architecture:** SE embedded *inside* each residual branch (not post-hoc)
- Equation 1: X̂ = X + SE(F(X))
- Equation 2: s = σ(W₂·δ(W₁·GAP(X))), X̂ = X·s, reduction r=16

**Runtime:** ~90 min on T4 GPU for full benchmark suite. Set `QUICK_RUN=True` to test with 5 epochs.

In [ ]:
# ─────────────────────────────────────────────
# CELL 1: Configuration
# ─────────────────────────────────────────────
QUICK_RUN   = False   # Set True for 5-epoch smoke test
EPOCHS      = 5 if QUICK_RUN else 100
BATCH_SIZE  = 64
SEED        = 42
LR          = 1e-4
WEIGHT_DECAY= 1e-5
PATIENCE    = 10      # early stopping patience
LR_PATIENCE = 5       # ReduceLROnPlateau patience
DROPOUT     = 0.5
SE_RATIO    = 16      # squeeze-and-excitation reduction ratio
GRAD_CLIP   = 5.0
NUM_CLASSES = 7
IMG_SIZE    = 48
EMOTION_LABELS = ['Angry','Disgust','Fear','Happy','Sad','Surprise','Neutral']

print('Config set. QUICK_RUN =', QUICK_RUN)

In [ ]:
# ─────────────────────────────────────────────
# CELL 2: Install & Imports
# ─────────────────────────────────────────────
!pip install -q kaggle grad-cam torchinfo
!git clone -q https://github.com/YOUR_USERNAME/ressenet-fer.git repo_src && cp -r repo_src/src ./src || true

import os, random, warnings, time, json, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset, random_split
import torchvision.transforms as T
import torchvision.models as models
from torchinfo import summary

from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay)
from sklearn.utils.class_weight import compute_class_weight
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

warnings.filterwarnings('ignore')

# Reproducibility
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 3: FER2013 Dataset Download
# ─────────────────────────────────────────────
# Option A: Kaggle API (fastest)
# Upload your kaggle.json first, OR use Option B below

if os.path.exists('/root/.config/kaggle/kaggle.json') or os.path.exists('kaggle.json'):
    if os.path.exists('kaggle.json'):
        os.makedirs('/root/.config/kaggle', exist_ok=True)
        os.system('cp kaggle.json /root/.config/kaggle/kaggle.json')
        os.system('chmod 600 /root/.config/kaggle/kaggle.json')
    os.system('kaggle datasets download -d msambare/fer2013 --unzip -p ./fer2013_data')
    DATA_PATH = './fer2013_data/fer2013.csv'
    print('Downloaded via Kaggle API')
else:
    # Option B: Manual upload
    # 1. Download fer2013.csv from https://www.kaggle.com/datasets/msambare/fer2013
    # 2. Upload it here in Colab, then set:
    DATA_PATH = 'fer2013.csv'
    print('Expecting fer2013.csv in current directory.')
    print('Download from: https://www.kaggle.com/datasets/msambare/fer2013')
    print('Then upload via Files panel on the left.')

# Verify
if os.path.exists(DATA_PATH):
    df_raw = pd.read_csv(DATA_PATH)
    print(f'Loaded: {df_raw.shape[0]} rows, columns: {list(df_raw.columns)}')
    print(df_raw['Usage'].value_counts())
else:
    print('ERROR: fer2013.csv not found. Please upload it.')

In [ ]:
# ───────────────────────────────────────────
# CELL 4: Dataset & DataLoader (now in src/dataset.py)
# ───────────────────────────────────────────
from src import Config, load_fer2013, build_loaders

CFG = Config(quick_run=QUICK_RUN, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED,
             lr=LR, weight_decay=WEIGHT_DECAY, patience=PATIENCE, lr_patience=LR_PATIENCE,
             dropout=DROPOUT, se_ratio=SE_RATIO, grad_clip=GRAD_CLIP,
             num_classes=NUM_CLASSES, img_size=IMG_SIZE, emotion_labels=EMOTION_LABELS)

train_df, val_df, test_df = load_fer2013(DATA_PATH)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

train_loader, val_loader, test_loader, class_weights_tensor = build_loaders(
    train_df, val_df, test_df, config=CFG, device=DEVICE
)
print('Class weights:', class_weights_tensor.cpu().numpy().round(3))

# Class distribution
fig, ax = plt.subplots(figsize=(8,3))
counts = train_df['emotion'].value_counts().sort_index()
ax.bar(EMOTION_LABELS, counts.values, color='steelblue', edgecolor='white')
ax.set_title('Training Set Class Distribution', fontsize=12, fontweight='bold')
ax.set_ylabel('Count'); ax.set_xlabel('Emotion')
plt.tight_layout(); plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ───────────────────────────────────────────
# CELL 5: Architecture Definitions (now in src/models.py)
# ───────────────────────────────────────────
from src import SEModule, ResSEBlock, ResSENet, SEPostHocBlock, SEResNetPostHoc

# ── Quick sanity check ───────────────────────
model_check = ResSENet(num_classes=7).to(DEVICE)
x_dummy = torch.randn(2, 3, 48, 48).to(DEVICE)
out = model_check(x_dummy)
print(f'Output shape: {out.shape}  (should be [2, 7])')
total_params = sum(p.numel() for p in model_check.parameters())
print(f'Total parameters: {total_params/1e6:.2f}M')
# NOTE: this comes out to ~11.3M, not the ~4.2M figure in the original notebook's
# comment -- flagged in README, verify against the manuscript's reported count.
del model_check, x_dummy

# ── Sanity check: post-hoc has identical param count to ResSENet ──
_a = ResSENet(num_classes=7).to(DEVICE)
_b = SEResNetPostHoc(num_classes=7).to(DEVICE)
_pa = sum(p.numel() for p in _a.parameters())
_pb = sum(p.numel() for p in _b.parameters())
print(f'ResSENet params: {_pa/1e6:.3f}M | SE-ResNet (post-hoc) params: {_pb/1e6:.3f}M '
      f'| iso-parametric: {_pa == _pb}')
del _a, _b

In [ ]:
# ───────────────────────────────────────────
# CELL 6: Baseline Architectures (now in src/models.py)
# ───────────────────────────────────────────
from src import (
    BaselineCNN, get_resnet18, get_vggnet13, get_resnet50,
    get_efficientnet_b0, get_ressenet_pretrained,
)

print('Baseline architectures loaded from src/models.py')
for name, fn, kw in [
    ('BaselineCNN',   BaselineCNN,         {}),
    ('ResNet-18',     get_resnet18,        {}),
    ('VGGNet-13',     get_vggnet13,        {}),
    ('ResSENet',      ResSENet,            {}),
    ('ResNet-50',     get_resnet50,        {'pretrained': True}),
    ('EfficientNet',  get_efficientnet_b0, {'pretrained': True}),
    ('ResSENet+',     get_ressenet_pretrained, {}),
]:
    m = fn(**kw).to(DEVICE)
    p = sum(x.numel() for x in m.parameters())
    print(f'  {name:<20} {p/1e6:.2f}M params')
    del m

In [ ]:
# ───────────────────────────────────────────
# CELL 7: Training & Evaluation Utilities (now in src/train.py)
# ───────────────────────────────────────────
from src import train_one_epoch, evaluate, train_model
print('Training utilities loaded from src/train.py')

In [ ]:
# ─────────────────────────────────────────────
# CELL 8: RUN FULL BENCHMARK (Table I)
# ─────────────────────────────────────────────
# ~90 min on T4 for full run, ~8 min for QUICK_RUN

BENCHMARK_CONFIGS = [
    ('Baseline CNN',       lambda: BaselineCNN(),                    False),
    ('ResNet-18',          lambda: get_resnet18(pretrained=False),   False),
    ('VGGNet-13',          lambda: get_vggnet13(pretrained=False),   False),
    ('SE-ResNet (post-hoc)', lambda: SEResNetPostHoc(),               False),  # iso-parametric control (Reviewer 1/2)
    ('ResSENet (Proposed)',lambda: ResSENet(),                       False),
    ('ResNet-50',          lambda: get_resnet50(pretrained=True),    True),
    ('EfficientNet-B0',    lambda: get_efficientnet_b0(pretrained=True), True),
    ('ResSENet+ (Pretrained)', lambda: get_resseNet_pretrained(),    True),
]

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
results   = {}
histories = {}

for model_name, model_fn, is_pretrained in BENCHMARK_CONFIGS:
    print(f'\n===== {model_name} =====')
    set_seed(SEED)
    model = model_fn().to(DEVICE)
    param_count = sum(p.numel() for p in model.parameters()) / 1e6

    model, hist = train_model(model, train_loader, val_loader, class_weights_tensor,
                              device=DEVICE, name=model_name, config=CFG)

    _, val_acc, _, _   = evaluate(model, val_loader,  criterion, DEVICE)
    _, test_acc, preds, labels = evaluate(model, test_loader, criterion, DEVICE)

    results[model_name] = {
        'pretrained': is_pretrained,
        'val_acc':   round(val_acc, 1),
        'test_acc':  round(test_acc, 1),
        'params_M':  round(param_count, 1),
        'preds':     preds,
        'labels':    labels,
    }
    histories[model_name] = hist

    # Save checkpoint
    torch.save(model.state_dict(), f"{model_name.replace(' ', '_')}.pt")
    del model
    torch.cuda.empty_cache()

print('\n===== Benchmark Complete =====')

In [ ]:
# ─────────────────────────────────────────────
# CELL 9: Print Table I (Paper Format)
# ─────────────────────────────────────────────
baseline_test = results['Baseline CNN']['test_acc']

print('\nTABLE I: Comparative Evaluation on FER2013')
print(f"{'':<30} {'Pretrain':<12} {'Val Acc':<12} {'Test Acc':<12} {'Params(M)':<12} {'Delta':<10}")
print('-' * 90)
for name, r in results.items():
    delta = r['test_acc'] - baseline_test
    pre = 'ImageNet' if r['pretrained'] else 'None'
    marker = ' <-- PROPOSED' if 'Proposed' in name else ''
    print(f"{name:<30} {pre:<12} {r['val_acc']:<12.1f} {r['test_acc']:<12.1f} {r['params_M']:<12.1f} +{delta:.1f}%{marker}")

# Save to CSV
table1_rows = []
for name, r in results.items():
    table1_rows.append({
        'Model': name,
        'Pretrained': 'ImageNet' if r['pretrained'] else 'None',
        'Val_Acc_%': r['val_acc'],
        'Test_Acc_%': r['test_acc'],
        'Params_M': r['params_M'],
        'Delta_vs_Baseline': round(r['test_acc'] - baseline_test, 1)
    })
pd.DataFrame(table1_rows).to_csv('table1_comparative.csv', index=False)
print('\nSaved: table1_comparative.csv')

In [ ]:
# ─────────────────────────────────────────────
# CELL 9b: Computational Complexity Metrics (Reviewer 2, Major Comment)
# FLOPs, parameter count, GPU latency, CPU latency, and throughput (FPS)
# for every model in Table I, measured at the paper's actual input size.
# ─────────────────────────────────────────────
import copy as _copy

N_WARMUP   = 10     # warm-up iterations (excluded from timing, avoids cold-start bias)
N_TIMED    = 100    # timed iterations
LATENCY_BATCH = 1    # single-image latency, standard for real-time inference reporting

complexity_results = {}

for model_name, model_fn, is_pretrained in BENCHMARK_CONFIGS:
    print(f'Profiling {model_name} ...')
    set_seed(SEED)
    model = model_fn().to(DEVICE)
    model.eval()

    # --- FLOPs (multiply-accumulate operations) via torchinfo ---
    try:
        info = summary(model, input_size=(1, 3, IMG_SIZE, IMG_SIZE),
                        device=DEVICE, verbose=0)
        macs = info.total_mult_adds
        flops = macs * 2  # 1 MAC = 2 FLOPs (standard convention)
    except Exception as e:
        print(f'  torchinfo FLOPs failed for {model_name}: {e}')
        flops = float('nan')

    param_count = sum(p.numel() for p in model.parameters())

    # --- GPU latency / throughput ---
    dummy = torch.randn(LATENCY_BATCH, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    with torch.no_grad():
        for _ in range(N_WARMUP):
            _ = model(dummy)
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(N_TIMED):
            _ = model(dummy)
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        t1 = time.time()
    gpu_latency_ms = (t1 - t0) / N_TIMED * 1000
    gpu_fps = 1000.0 / gpu_latency_ms

    # --- CPU latency / throughput (deployment-relevant for edge use case) ---
    model_cpu = _copy.deepcopy(model).to('cpu')
    dummy_cpu = torch.randn(LATENCY_BATCH, 3, IMG_SIZE, IMG_SIZE)
    with torch.no_grad():
        for _ in range(min(N_WARMUP, 5)):
            _ = model_cpu(dummy_cpu)
        t0 = time.time()
        for _ in range(min(N_TIMED, 30)):  # fewer iters on CPU to keep runtime reasonable
            _ = model_cpu(dummy_cpu)
        t1 = time.time()
    n_cpu_iters = min(N_TIMED, 30)
    cpu_latency_ms = (t1 - t0) / n_cpu_iters * 1000
    cpu_fps = 1000.0 / cpu_latency_ms

    complexity_results[model_name] = {
        'params_M':        round(param_count / 1e6, 3),
        'flops_M':         round(flops / 1e6, 2) if flops == flops else None,  # NaN check
        'gpu_latency_ms':  round(gpu_latency_ms, 3),
        'gpu_fps':         round(gpu_fps, 1),
        'cpu_latency_ms':  round(cpu_latency_ms, 3),
        'cpu_fps':         round(cpu_fps, 1),
    }

    del model, model_cpu, dummy, dummy_cpu
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# --- Print as a paper-ready table ---
print()
print(f'DEVICE used for GPU timing: {DEVICE}')
print('TABLE: Computational Complexity (input size {0}x{0}, batch=1)'.format(IMG_SIZE))
print(f"{'Model':<26}{'Params(M)':<12}{'FLOPs(M)':<12}{'GPU ms':<10}{'GPU FPS':<10}{'CPU ms':<10}{'CPU FPS':<10}")
print('-' * 90)
for name, r in complexity_results.items():
    print(f"{name:<26}{r['params_M']:<12}{r['flops_M']:<12}{r['gpu_latency_ms']:<10}"
          f"{r['gpu_fps']:<10}{r['cpu_latency_ms']:<10}{r['cpu_fps']:<10}")

pd.DataFrame([{'Model': k, **v} for k, v in complexity_results.items()]).to_csv(
    'table_complexity_metrics.csv', index=False)
print('\nSaved: table_complexity_metrics.csv')
print('NOTE: latency/FPS numbers are hardware-specific -- record the exact GPU/CPU')
print('      (e.g. Colab T4, or "Intel Xeon (Colab CPU runtime)") in the paper\'s')
print('      Methods section alongside these numbers, since reviewers may ask.')


In [ ]:
# ─────────────────────────────────────────────
# CELL 9c: Training Dynamics -- ResSENet vs. SE-ResNet (post-hoc)
# (Reviewer 2, Major Comment: convergence speed / stability comparison
#  between the two SE-placement variants that isolate the paper's core claim)
# ─────────────────────────────────────────────
import numpy as _np

pair_names = ['SE-ResNet (post-hoc)', 'ResSENet (Proposed)']
missing = [n for n in pair_names if n not in histories]
if missing:
    print(f'WARNING: {missing} not found in `histories` -- re-run Cell 8 first.')
else:
    dynamics_summary = {}
    for name in pair_names:
        h = histories[name]
        val_losses = h['val_loss']
        val_accs   = h['val_acc']
        best_epoch = int(_np.argmin(val_losses)) + 1
        # "converged" = first epoch within 1% (absolute) of final best val acc
        best_acc = max(val_accs)
        converge_epoch = next((i + 1 for i, a in enumerate(val_accs) if a >= best_acc - 1.0),
                               len(val_accs))
        # stability: std of val loss over the last 10 epochs actually trained
        tail = val_losses[-10:] if len(val_losses) >= 10 else val_losses
        dynamics_summary[name] = {
            'epochs_trained':        len(val_losses),
            'epoch_of_best_val_loss': best_epoch,
            'epoch_to_converge':      converge_epoch,
            'best_val_acc':           round(best_acc, 2),
            'final_val_loss_std_last10': round(float(_np.std(tail)), 4),
        }

    print('TABLE: Training Dynamics Comparison')
    print(f"{'Metric':<32}{'SE-ResNet (post-hoc)':<24}{'ResSENet (Proposed)':<20}")
    print('-' * 76)
    for metric in ['epochs_trained', 'epoch_of_best_val_loss', 'epoch_to_converge',
                   'best_val_acc', 'final_val_loss_std_last10']:
        v1 = dynamics_summary[pair_names[0]][metric]
        v2 = dynamics_summary[pair_names[1]][metric]
        print(f"{metric:<32}{str(v1):<24}{str(v2):<20}")

    pd.DataFrame(dynamics_summary).T.to_csv('table_training_dynamics.csv')
    print('\nSaved: table_training_dynamics.csv')

    # Plot val loss curves for both, for a qualitative "gradient flow /
    # stability" figure if the paper wants one
    plt.figure(figsize=(7, 4.5))
    for name in pair_names:
        plt.plot(histories[name]['val_loss'], label=name)
    plt.xlabel('Epoch')
    plt.ylabel('Validation Loss')
    plt.title('Validation Loss: Post-Hoc SE vs. Pre-Addition SE (ResSENet)')
    plt.legend()
    plt.tight_layout()
    plt.savefig('training_dynamics_comparison.png', dpi=150)
    plt.show()
    print('Saved: training_dynamics_comparison.png')


In [ ]:
# ─────────────────────────────────────────────
# CELL 10: Ablation Study (Table II)
# ─────────────────────────────────────────────
ABLATION_CONFIGS = [
    ('(i) Baseline CNN',         False, False),
    ('(ii) CNN + Residual',      True,  False),
    ('(iii) CNN + SE Only',      False, True),
    ('(iv) ResSENet (Proposed)', True,  True),
]

ablation_results = {}

for abl_name, use_res, use_se in ABLATION_CONFIGS:
    # (i) reuse baseline CNN result to save time
    if abl_name == '(i) Baseline CNN' and 'Baseline CNN' in results:
        ablation_results[abl_name] = {
            'residual': False, 'se': False,
            'val_acc': results['Baseline CNN']['val_acc'],
            'test_acc': results['Baseline CNN']['test_acc'],
        }
        continue
    if abl_name == '(iv) ResSENet (Proposed)' and 'ResSENet (Proposed)' in results:
        ablation_results[abl_name] = {
            'residual': True, 'se': True,
            'val_acc': results['ResSENet (Proposed)']['val_acc'],
            'test_acc': results['ResSENet (Proposed)']['test_acc'],
        }
        continue

    print(f'\n===== ABLATION: {abl_name} =====')
    set_seed(SEED)
    model = ResSENet(use_residual=use_res, use_se=use_se).to(DEVICE)
    model, _ = train_model(model, name=abl_name, epochs=EPOCHS)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    _, val_acc, _, _  = evaluate(model, val_loader, criterion)
    _, test_acc, _, _ = evaluate(model, test_loader, criterion)
    ablation_results[abl_name] = {
        'residual': use_res, 'se': use_se,
        'val_acc': round(val_acc, 1), 'test_acc': round(test_acc, 1)
    }
    del model; torch.cuda.empty_cache()

print('\nTABLE II: Ablation Study on FER2013')
print(f"{'':<30} {'Residual':<12} {'SE':<10} {'Val Acc':<12} {'Test Acc'}")
print('-' * 75)
for abl_name, r in ablation_results.items():
    res_str = 'YES' if r['residual'] else 'NO'
    se_str  = 'YES' if r['se'] else 'NO'
    print(f"{abl_name:<30} {res_str:<12} {se_str:<10} {r['val_acc']:<12.1f} {r['test_acc']:.1f}")

pd.DataFrame([
    {'Config': k, **{kk: vv for kk, vv in v.items() if kk != 'preds'}}
    for k, v in ablation_results.items()
]).to_csv('table2_ablation.csv', index=False)
print('\nSaved: table2_ablation.csv')

In [ ]:
# ─────────────────────────────────────────────
# CELL 11: Training Curves for ResSENet
# ─────────────────────────────────────────────
hist = histories.get('ResSENet (Proposed)', None)
if hist:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist['train_loss'], label='Train', linewidth=1.5)
    axes[0].plot(hist['val_loss'],   label='Validation', linewidth=1.5)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('ResSENet: Loss Curve', fontweight='bold')
    axes[0].legend()

    axes[1].plot(hist['train_acc'], label='Train', linewidth=1.5)
    axes[1].plot(hist['val_acc'],   label='Validation', linewidth=1.5)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('ResSENet: Accuracy Curve', fontweight='bold')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved: training_curves.png')

In [ ]:
# ─────────────────────────────────────────────
# CELL 12: Confusion Matrix + Per-Class Report
# ─────────────────────────────────────────────
res_key = 'ResSENet (Proposed)'
preds_res  = results[res_key]['preds']
labels_res = results[res_key]['labels']

cm = confusion_matrix(labels_res, preds_res)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
ConfusionMatrixDisplay(cm, display_labels=EMOTION_LABELS).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Normalized
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS,
            ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png\n')

print('Per-class Classification Report (ResSENet):')
print(classification_report(labels_res, preds_res,
                             target_names=EMOTION_LABELS, digits=3))

In [ ]:
# ─────────────────────────────────────────────
# CELL 13: Accuracy-Parameter Trade-Off Plot
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

colors = {'None': '#2196F3', 'ImageNet': '#FF5722'}
for name, r in results.items():
    pre = 'ImageNet' if r['pretrained'] else 'None'
    is_ours = ('Proposed' in name) or ('ResSENet+' in name)
    size = 250 if is_ours else 120
    marker = '*' if is_ours else 'o'
    ax.scatter(r['params_M'], r['test_acc'],
               s=size, c=colors[pre], marker=marker,
               edgecolors='black', linewidths=0.8, zorder=5)
    offset = (0.3, 0.4) if 'ResSENet' in name else (0.3, -0.7)
    short = name.replace(' (Proposed)', '').replace(' (Pretrained)', '+')
    ax.annotate(short, (r['params_M'], r['test_acc']),
                textcoords='offset points', xytext=(8, 4),
                fontsize=8.5)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#2196F3',
           markeredgecolor='black', markersize=9, label='From Scratch'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#FF5722',
           markeredgecolor='black', markersize=9, label='ImageNet Pretrained'),
    Line2D([0],[0], marker='*', color='w', markerfacecolor='gray',
           markeredgecolor='black', markersize=13, label='ResSENet (Ours)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
ax.set_xlabel('Parameters (Millions)', fontsize=11)
ax.set_ylabel('Test Accuracy (%)', fontsize=11)
ax.set_title('Accuracy vs. Parameter Count on FER2013', fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('accuracy_param_tradeoff.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: accuracy_param_tradeoff.png')

In [ ]:
# ─────────────────────────────────────────────
# CELL 14: GradCAM Interpretability
# ─────────────────────────────────────────────
# Reload best ResSENet checkpoint
resseNet_model = ResSENet(num_classes=NUM_CLASSES).to(DEVICE)
ckpt_path = 'ResSENet_(Proposed).pt'
if os.path.exists(ckpt_path):
    resseNet_model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    print('Loaded checkpoint.')
else:
    print('Checkpoint not found; using current model weights.')
resseNet_model.eval()

target_layer = [resseNet_model.stage4[-1].conv2]
cam = GradCAM(model=resseNet_model, target_layers=target_layer)

# Grab one test sample per class
samples_per_class = {}
for idx in range(len(test_ds)):
    img_t, lbl = test_ds[idx]
    if lbl not in samples_per_class:
        samples_per_class[lbl] = img_t
    if len(samples_per_class) == NUM_CLASSES:
        break

fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(NUM_CLASSES * 2.2, 5))
fig.suptitle('GradCAM Interpretability: Final Conv Layer Activations', fontweight='bold', fontsize=12)

for cls_idx in range(NUM_CLASSES):
    img_t = samples_per_class.get(cls_idx)
    if img_t is None:
        continue
    input_tensor = img_t.unsqueeze(0).to(DEVICE)

    # Raw image (denormalize)
    raw = img_t.numpy().transpose(1, 2, 0)
    raw = raw * SIGMA + MU
    raw = np.clip(raw, 0, 1).astype(np.float32)

    grayscale_cam = cam(input_tensor=input_tensor,
                        targets=[ClassifierOutputTarget(cls_idx)])
    cam_img = show_cam_on_image(raw, grayscale_cam[0], use_rgb=True)

    axes[0][cls_idx].imshow(raw)
    axes[0][cls_idx].set_title(EMOTION_LABELS[cls_idx], fontsize=9)
    axes[0][cls_idx].axis('off')

    axes[1][cls_idx].imshow(cam_img)
    axes[1][cls_idx].axis('off')

axes[0][0].set_ylabel('Input', fontsize=9)
axes[1][0].set_ylabel('GradCAM', fontsize=9)
plt.tight_layout()
plt.savefig('gradcam_visualization.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: gradcam_visualization.png')

In [ ]:
# ─────────────────────────────────────────────
# CELL 15: Ablation Bar Chart
# ─────────────────────────────────────────────
abl_names  = list(ablation_results.keys())
abl_test   = [ablation_results[k]['test_acc'] for k in abl_names]
colors_abl = ['#90A4AE','#42A5F5','#66BB6A','#EF5350']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(range(len(abl_names)), abl_test,
              color=colors_abl, edgecolor='black', linewidth=0.8)
for bar, acc in zip(bars, abl_test):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(range(len(abl_names)))
ax.set_xticklabels([n.split('\n')[0] for n in abl_names], rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Test Accuracy (%)', fontsize=11)
ax.set_title('Table II: Ablation Study Results (FER2013)', fontweight='bold', fontsize=12)
ax.set_ylim(55, 75)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('ablation_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: ablation_chart.png')

In [ ]:
# ─────────────────────────────────────────────
# CELL 16: Model Summary (for Methods section)
# ─────────────────────────────────────────────
set_seed(SEED)
m = ResSENet(num_classes=NUM_CLASSES).to(DEVICE)
summary(m, input_size=(1, 3, IMG_SIZE, IMG_SIZE), device=DEVICE,
        col_names=['input_size','output_size','num_params','trainable'])
del m

In [ ]:
# ─────────────────────────────────────────────
# CELL 17: Save All Results as JSON
# ─────────────────────────────────────────────
export = {}
for k, v in results.items():
    export[k] = {kk: vv for kk, vv in v.items() if kk not in ('preds', 'labels')}

export['ablation'] = {
    k: {kk: vv for kk, vv in v.items() if kk not in ('preds', 'labels')}
    for k, v in ablation_results.items()
}

with open('resSENet_all_results.json', 'w') as f:
    json.dump(export, f, indent=2)

print('All results saved to resSENet_all_results.json')
print(json.dumps({k: {'val': v['val_acc'], 'test': v['test_acc']}
                  for k, v in export.items() if k != 'ablation'}, indent=2))

## Summary of Outputs

| File | Used For |
|---|---|
| `table1_comparative.csv` | Table I in paper |
| `table2_ablation.csv` | Table II in paper |
| `table_complexity_metrics.csv` | FLOPs/latency/FPS (Reviewer 2, complexity metrics) |
| `table_training_dynamics.csv` | Convergence/stability comparison (Reviewer 2, training dynamics) |
| `training_dynamics_comparison.png` | Val-loss curve, post-hoc SE vs. ResSENet |
| `training_curves.png` | Learning curve figure |
| `confusion_matrix.png` | Per-class analysis figure |
| `accuracy_param_tradeoff.png` | Fig. 3 in paper |
| `gradcam_visualization.png` | GradCAM interpretability figure |
| `ablation_chart.png` | Ablation bar chart |
| `resSENet_all_results.json` | Full reproducibility record |
| `*.pt` | Model checkpoints |

### Key Results (as reported in the published paper)
- **ResSENet (from scratch):** 65.3% test accuracy on FER2013, 11.3M parameters
- **vs ResNet-18:** +6.5% at a comparable parameter budget (11.3M vs 11.2M)
- **vs VGGNet-13:** +1.0% with 91% fewer parameters (11.3M vs 129.0M)
- **vs post-hoc SE-ResNet (iso-parametric control):** +0.6%, 22% faster convergence, 36% lower GPU latency (3.84 ms vs 5.22 ms)
- **Ablation:** residual connections and SE together give +14.0% over the baseline
